# Sarcasm Detection Using Traditional Machine Learning

Classification of news headlines as sarcastic (`1`) or non-sarcastic (`0`) using TF-IDF features with Logistic Regression and a Linear SVM, plus an investigation of publisher-label confounding. Based on Rishabh Misra's News Headlines Dataset for Sarcasm Detection.

In [ ]:
# Colab: uncomment if needed
# !pip install datasets scikit-learn pandas numpy matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import joblib

## 1. Load the dataset

The Hugging Face dataset mirrors the original 26,709-headline corpus with `article_link`, `headline` and `is_sarcastic`.

In [ ]:
ds = load_dataset("sagnikrayc/news-headlines-dataset-for-sarcasm-detection")
df = pd.DataFrame(ds["train"])
df = df.rename(columns={"headline": "headline", "is_sarcastic": "is_sarcastic"})
df[["article_link", "headline", "is_sarcastic"]].head()

## 2. Data preparation

Keep headline text and label, check for missing values, drop duplicate headlines and remove conflicting labels.

In [ ]:
data = df[["headline", "is_sarcastic"]].dropna().copy()
counts = data.groupby("headline")["is_sarcastic"].nunique()
conflicting = set(counts[counts > 1].index)
data = data[~data["headline"].isin(conflicting)]
data = data.drop_duplicates(subset="headline").reset_index(drop=True)
print(f"Cleaned headlines: {len(data)}")
print(data["is_sarcastic"].value_counts())

## 3. Stratified train/test split (80/20)

In [ ]:
X = data["headline"]
y = data["is_sarcastic"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(len(X_train), len(X_test))

## 4. Models: TF-IDF (1-2 grams) + Logistic Regression / Linear SVM

TF-IDF uses `ngram_range=(1, 2)`, `min_df=2`, `max_features=50000`, sublinear TF.

In [ ]:
def make_vectorizer():
    return TfidfVectorizer(
        ngram_range=(1, 2), min_df=2, max_features=50000,
        sublinear_tf=True, strip_accents="unicode"
    )

models = {
    "Logistic Regression": Pipeline([
        ("tfidf", make_vectorizer()),
        ("clf", LogisticRegression(max_iter=1000))
    ]),
    "Linear SVM": Pipeline([
        ("tfidf", make_vectorizer()),
        ("clf", LinearSVC())
    ]),
}

## 5. Held-out evaluation

In [ ]:
rows = []
fitted = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted[name] = model
    pred = model.predict(X_test)
    rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred),
    })
    print(name, "confusion matrix:\n", confusion_matrix(y_test, pred))
pd.DataFrame(rows).set_index("model")

## 6. Five-fold stratified cross-validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {"accuracy": "accuracy", "precision": "precision", "recall": "recall", "f1": "f1"}
for name, model in models.items():
    scores = cross_validate(model, X, y, cv=cv, scoring=scoring)
    print(f"\n{name}")
    for metric in scoring:
        vals = scores[f"test_{metric}"]
        print(f"  {metric}: {vals.mean():.4f} +/- {vals.std():.4f}")

## 7. Publisher-label bias investigation

Huffington Post headlines are all non-sarcastic; The Onion headlines are all sarcastic. A publisher-only classifier therefore reaches 100% accuracy, which invalidates any claim that the headline model learned sarcasm itself.

In [ ]:
pubs = df[["article_link", "is_sarcastic"]].copy()
pubs["publisher"] = pubs["article_link"].str.extract(r"://(?:www\.)?([^/]+)")
print(pd.crosstab(pubs["publisher"], pubs["is_sarcastic"]).head(10))

Xp = pd.get_dummies(pubs["publisher"])
pub_model = LogisticRegression(max_iter=1000)
Xp_train, Xp_test, yp_train, yp_test = train_test_split(
    Xp, pubs["is_sarcastic"], test_size=0.2, random_state=42, stratify=pubs["is_sarcastic"]
)
pub_model.fit(Xp_train, yp_train)
print("Publisher-only accuracy:", accuracy_score(yp_test, pub_model.predict(Xp_test)))
print("Majority baseline:", max(pubs["is_sarcastic"].mean(), 1 - pubs["is_sarcastic"].mean()))

## 8. Persist the preferred pipeline

Save the Linear SVM pipeline for the Streamlit app in `app.py`.

In [ ]:
# Run from the sarcasm-detection directory
# joblib.dump(fitted["Linear SVM"], "models/linear_svm.pkl")
# joblib.dump(fitted["Logistic Regression"], "models/logistic_regression.pkl")

## Limitations

Publisher identity is perfectly confounded with the label; TF-IDF cannot capture context, tone or implied meaning; random splitting does not test generalisation to unseen publishers. Results must not be read as general-purpose sarcasm understanding.